In [ ]:
import deeptime as dpt
from tqdm.notebook import tqdm # for progress bar
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pyemma as pem
import pickle
import os

In [ ]:
# Step 1: Load your Q-G data
data = np.load("QG.npy")  # shape (50, 13333, 2)

# Test how many percent of variation that Q and G account for
scaler = StandardScaler()
raw_data_all = np.concatenate(data, axis=0)
scaler.fit(raw_data_all)
processed_data = [scaler.transform(traj) for traj in data]  # List of 2D arrays
conc_processed_data = np.concatenate(processed_data, axis=0)

tica = dpt.decomposition.TICA(lagtime=1, dim=None)
tica_model = tica.fit(processed_data)
projected = tica_model.transform(processed_data)

print(tica_model.model.cumulative_kinetic_variance)

In [ ]:

"""
In general, I want to do MSM on QG space. However, some proteins do not sample the entangled state.
Hence, in that case we need to do MSM on Q space alone.
By defaults, n_dims is 2- mean perform on QG space. If not working, then set to n_dims=1 to use Q only
"""
# Set the number of dimensions to use for the MSM
n_dims = 2

if n_dims == 2: 
    # Step 2: Standardize the data
    scaler = StandardScaler()
    raw_data_all = np.concatenate(data, axis=0)
    scaler.fit(raw_data_all)
    processed_data = [scaler.transform(traj) for traj in data]  # List of 2D arrays
    conc_processed_data = np.concatenate(processed_data, axis=0)

elif n_dims == 1:
    # Incase MSM is not working on GQ space since the G is not significant, did MSM on Q space alone
    # Step 2: Load your Q-G data
    data = data[:, :, 0][:, :, np.newaxis]  # shape becomes (50, 13333, 1)
    # Step 2: Standardize the data
    scaler = StandardScaler()
    raw_data_all = np.concatenate(data, axis=0)
    scaler.fit(raw_data_all)
    processed_data = [scaler.transform(traj) for traj in data]  # List of 2D arrays
    conc_processed_data = np.concatenate(processed_data, axis=0)

In [ ]:
# Use KMeans from deeptime 
n_clusters = 100
kmeans_model = dpt.clustering.KMeans(n_clusters=n_clusters,
                                    max_iter=500,
                                    fixed_seed=True,
                                    progress=tqdm
                                    ).fit_fetch(processed_data)

# Transform each trajectory to discrete state trajectory
discrete_trajectories = [kmeans_model.transform(traj) for traj in processed_data]

# Get cluster centers (in scaled space)
cluster_centers_scaled = kmeans_model.cluster_centers

# Inverse transform to original Q-G scale
cluster_centers_original = scaler.inverse_transform(cluster_centers_scaled)

# Optional: Check shapes and values
print("Number of trajectories:", len(discrete_trajectories))
print("Shape of first discrete trajectory:", discrete_trajectories[0].shape)
print("Unique states in first trajectory:", np.unique(discrete_trajectories[0]))

In [ ]:
# Optional: Bayesian MSM with multiple lag times to find the optimal lag time
models = []
lagtimes = np.arange(1, 101, 10)  # e.g., 1, 6, 11, ..., 46

for lag in tqdm(lagtimes, desc="Fitting Bayesian MSMs"):
    # Estimate transition counts
    counts = dpt.markov.TransitionCountEstimator(lagtime=lag, count_mode='effective').fit_fetch(discrete_trajectories)

    # Fit Bayesian MSM
    msm = dpt.markov.msm.BayesianMSM(n_samples=10).fit_fetch(counts)

    # Store model for ITS calculation
    models.append(msm)

# Compute implied timescales from fitted MSMs
its_data = dpt.util.validation.implied_timescales(models)

# Plotting the top 5 implied timescales
fig, ax = plt.subplots(figsize=(8, 6))
dpt.plots.plot_implied_timescales(its_data, n_its=5, ax=ax)

ax.set_yscale('log')
ax.set_title('Implied Timescales vs Lag Time')
ax.set_xlabel('Lag time (frames)')
ax.set_ylabel('Timescale (frames)')
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Optional: Plot the CK test
posterior = models[3] # lagtime = 31
# ck_result = dpt.util.validation.ck_test(models, n_metastable_sets=4)
ck_result = posterior.ck_test(models, n_metastable_sets=4)
dpt.plots.plot_ck_test(ck_result)

In [ ]:
# Choose a lag time (in frames) — determined from the implied timescales plot.
# currently, we use the lagtime = 1, which we do not care about the kinetics of the system.
lagtime = 1  # or 20, etc.

# Fit MSM
"""
The parameters allow_disconnected=False means that the MSM is not allowed to have disconnected states (default value in deeptime).
If set to True, the MSM will have disconnected states because the transition matrix may have disconnected and transient states,
and the estimated stationary distribution is only meaningful on the respective connected set.
"""
msm = dpt.markov.msm.MaximumLikelihoodMSM(lagtime=lagtime, allow_disconnected=False).fit(discrete_trajectories).fetch_model()

# Print some basic info
print("Number of microstates:", msm.transition_matrix.shape[0]) #equivalent to msm.n_states
print("Top 5 eigenvalues:", msm.eigenvalues()[:5])
print("Top 5 relaxation timescales:", msm.timescales()[:5])

# Print some basic info
print("Top 10 eigenvalues and timescales:")
print("Top 10 eigenvalues:", msm.eigenvalues()[:10])
print("Top 10 relaxation timescales:", msm.timescales()[:10])

In [ ]:
nbins=30

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 4))

# Plot the global landscape
pem.plots.plot_free_energy(raw_data_all[:, 0], raw_data_all[:, 1], nbins=nbins, ax=ax)
ax.set_ylim([0, np.max(raw_data_all[:, 1])])
ax.set_xlim([0, 1])
ax.set_title("Global")

plt.tight_layout()
plt.show()

In [ ]:
# Perform PCCA
n_macrostates = 5 # can be determined from the implied timescales plot.
pcca = msm.pcca(n_macrostates)

# Hard assignment: each cluster assigned to a macrostate
macrostate_assignment = pcca.assignments  # shape (n_microstates,)
print("Macrostate assignments (first 10):", macrostate_assignment[:10])

# Soft assignment: probability each microstate belongs to each macrostate
membership = pcca.memberships  # shape (n_microstates, n_macrostates)
print("Membership matrix shape:", membership.shape)

"""
Incase you want to see the assignment of the microstates to the macrostates
The assignment of the microstates to the macrostates
for each cluster from Kmeans clustering (order in the array, which macrostate it belongs to (value in the array))

The pcca.sets is the assignment of the macrostates to the macrostates
for each macrostate from PCCA (order in the array, which macrostate it belongs to (value in the array))
"""
print(f"The assignment of the microstates to the macrostates: {macrostate_assignment}")
print("The assignment of the macrostates to the macrostates:")
for idx, pcca_set in enumerate(pcca.sets):
    print(f"Macrostate {idx}: {pcca_set}")

In [ ]:
# within each macrostate, what is the microstate with the highest probability?
most_prob_microstate = np.argmax(membership, axis=0)
# location of the most prob microstate in the original data
for macro_idx, micro_idx in enumerate(most_prob_microstate):
    print(f"Macrostate {macro_idx}: {micro_idx}: {cluster_centers_original[micro_idx]}")

# because the macrostate indices from deeptime are not sorted for intutitive, we need to sort them. so that the smallest macrostate index is the most stable macrostate.
# corresponding to the state with the lowest Q.
macro_state_sorted = np.argsort(cluster_centers_original[most_prob_microstate][:,0])
macro_state_mapping = {}
for i in range(len(macro_state_sorted)):
    print(f"Macrostate old: {macro_state_sorted[i]} -> New: {i}")
    macro_state_mapping[macro_state_sorted[i]] = i

In [ ]:
# convert macrostate_assignment to the sorted macrostate_assignment
macrostate_assignment_sorted = np.zeros_like(macrostate_assignment)
for i in range(len(macrostate_assignment)):
    macrostate_assignment_sorted[i] = macro_state_mapping[macrostate_assignment[i]]

# convert the microstates to macrostates
meta_dtrajs = [[macrostate_assignment_sorted[i] for i in dtraj] for dtraj in discrete_trajectories]
meta_dtrajs_flat = np.array(np.array(meta_dtrajs).flatten(), dtype=int)

In [ ]:
# Check state probability along the whole trajectory
for i in range(n_macrostates):
    print(f"Macrostate {i}: {len(meta_dtrajs_flat[meta_dtrajs_flat==i])/len(meta_dtrajs_flat):.3f}")

In [ ]:
# Plot the free energy surface and the state map
x = np.array(raw_data_all[:,0])
y = np.array(raw_data_all[:,1])

nbins = 50 # number of bins for the free energy surface and the state map- Protein-specific

# Create combined figure with two panels using GridSpec
fig = plt.figure(figsize=(13, 7))
gw = int(np.floor(0.5 + 500 * fig.get_figwidth()))
gh = int(np.floor(0.5 + 500 * fig.get_figheight()))
gs = plt.GridSpec(gh, gw)
gs.update(hspace=0.0, wspace=0.0, left=0.0, right=1.0, bottom=0.0, top=1.0)
ax_box = fig.add_subplot(gs[:, :])
ax_box.set_axis_off()

# Panel A: Free Energy Surface (left)
ax_fe = fig.add_subplot(gs[500:2750, 500:3000])
# Plot on the raw data
_, _, misc = pem.plots.plot_free_energy(x, y, ax=ax_fe, nbins=nbins,
                                        cax=fig.add_subplot(gs[300:400, 500:3000]),
                                        cbar_orientation='horizontal',
                                        cmap='coolwarm',
                                        levels=100,
                                        legacy=False
                                        )

# # if need, plot the cluster centers
# ax_fe.scatter(
#        cluster_centers_original[:, 0],  # Q
#        cluster_centers_original[:, 1],  # G
#        c='k', marker='o', s=40, label='Cluster Centers'
#    )

misc['cbar'].ax.xaxis.set_ticks_position('top')
misc['cbar'].ax.xaxis.set_label_position('top')
misc['cbar'].set_label(r'-ln(P) / $\mathrm{k}_\mathrm{B}T$')

ax_fe.set_ylabel('$G$', fontsize=16)
ax_fe.set_xlabel('$Q$', fontsize=16)
ax_fe.set_ylim([np.min(y), np.max(y)])

# Panel B: State Map (right)
ax_state = fig.add_subplot(gs[500:2750, 3500:6000])

_, _, misc = pem.plots.plot_state_map(x, y, meta_dtrajs_flat, ax=ax_state, nbins=nbins,
                                      cax=fig.add_subplot(gs[300:400, 3500:6000]),
                                      cbar_orientation='horizontal'
                                      )



misc['cbar'].ax.xaxis.set_ticks_position('top')
misc['cbar'].ax.xaxis.set_label_position('top')
misc['cbar'].set_label('Macrostate')


ax_state.set_ylabel('$G$', fontsize=16)
ax_state.set_xlabel('$Q$', fontsize=16)
ax_state.set_ylim([np.min(y), np.max(y)])
# ax_state.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Add panel labels
ax_fe.text(0.05, 0.95, 'A', transform=ax_fe.transAxes, fontsize=20, 
           fontweight='bold', verticalalignment='top')
ax_state.text(0.05, 0.95, 'B', transform=ax_state.transAxes, fontsize=20, 
              fontweight='bold', verticalalignment='top')

# plt.tight_layout()
plt.savefig('QG_MSM.png', dpi=600)
plt.show()

In [ ]:
# Check state probability along the last 200ns of the trajectory
meta_dtrajs_all = np.asarray(meta_dtrajs)
meta_dtrajs_last_200ns = meta_dtrajs_all[:,-2666:]
meta_dtrajs_last_200ns_flat = np.concatenate(meta_dtrajs_last_200ns)
for i in range(n_macrostates):
    print(f"Macrostate {i}: {len(meta_dtrajs_last_200ns_flat[meta_dtrajs_last_200ns_flat==i])/len(meta_dtrajs_last_200ns_flat):.3f}")

print("---------------\nNative state: \n")
native_index = np.max(meta_dtrajs_all)
print(f"{native_index=}")
print(f"{len(meta_dtrajs_last_200ns_flat[meta_dtrajs_last_200ns_flat==native_index])/len(meta_dtrajs_last_200ns_flat):.3f}")

In [ ]:
# Plot on the last 200 ns
# Plot the free energy surface and the state map
raw_data_all_last200ns = np.concatenate(data[:,-2666:, :], axis=0)

x = np.array(raw_data_all_last200ns[:,0])
y = np.array(raw_data_all_last200ns[:,1])


# nbins = 50 # number of bins for the free energy surface and the state map- Protein-specific

# Create combined figure with two panels using GridSpec
fig = plt.figure(figsize=(13, 7))
gw = int(np.floor(0.5 + 500 * fig.get_figwidth()))
gh = int(np.floor(0.5 + 500 * fig.get_figheight()))
gs = plt.GridSpec(gh, gw)
gs.update(hspace=0.0, wspace=0.0, left=0.0, right=1.0, bottom=0.0, top=1.0)
ax_box = fig.add_subplot(gs[:, :])
ax_box.set_axis_off()

# Panel A: Free Energy Surface (left)
ax_fe = fig.add_subplot(gs[500:2750, 500:3000])
# Plot on the raw data
_, _, misc = pem.plots.plot_free_energy(x, y, ax=ax_fe, nbins=nbins,
                                        cax=fig.add_subplot(gs[300:400, 500:3000]),
                                        cbar_orientation='horizontal',
                                        cmap='coolwarm',
                                        levels=100,
                                        legacy=False
                                        )

# # if need, plot the cluster centers
# ax_fe.scatter(
#        cluster_centers_original[:, 0],  # Q
#        cluster_centers_original[:, 1],  # G
#        c='k', marker='o', s=40, label='Cluster Centers'
#    )

misc['cbar'].ax.xaxis.set_ticks_position('top')
misc['cbar'].ax.xaxis.set_label_position('top')
misc['cbar'].set_label(r'-ln(P) / $\mathrm{k}_\mathrm{B}T$')

ax_fe.set_ylabel('$G$', fontsize=16)
ax_fe.set_xlabel('$Q$', fontsize=16)
ax_fe.set_ylim([np.min(y), np.max(y)])

# Panel B: State Map (right)
ax_state = fig.add_subplot(gs[500:2750, 3500:6000])

_, _, misc = pem.plots.plot_state_map(x, y, meta_dtrajs_last_200ns_flat, ax=ax_state, nbins=nbins,
                                      cax=fig.add_subplot(gs[300:400, 3500:6000]),
                                      cbar_orientation='horizontal'
                                      )



misc['cbar'].ax.xaxis.set_ticks_position('top')
misc['cbar'].ax.xaxis.set_label_position('top')
misc['cbar'].set_label('Macrostate')


ax_state.set_ylabel('$G$', fontsize=16)
ax_state.set_xlabel('$Q$', fontsize=16)
ax_state.set_ylim([np.min(y), np.max(y)])
# ax_state.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Add panel labels
ax_fe.text(0.05, 0.95, 'A', transform=ax_fe.transAxes, fontsize=20, 
           fontweight='bold', verticalalignment='top')
ax_state.text(0.05, 0.95, 'B', transform=ax_state.transAxes, fontsize=20, 
              fontweight='bold', verticalalignment='top')

# plt.tight_layout()
plt.savefig('QG_MSM_last200ns.png', dpi=600)
plt.show()

# Save the results

In [ ]:
# Check if 'msm_results' folder exists, create if not
results_dir = 'msm_results'
os.makedirs(results_dir, exist_ok=True)

# Save arrays
np.savez(os.path.join(results_dir, 'msm_analysis_results.npz'),
         cluster_centers_scaled=cluster_centers_scaled,
         cluster_centers_original=cluster_centers_original,
         discrete_trajectories=discrete_trajectories,
         macrostate_assignment=macrostate_assignment,
         macrostate_assignment_sorted=macrostate_assignment_sorted,
         membership=membership,
         processed_data=processed_data,
         meta_dtrajs=meta_dtrajs
)

if scaler is not None:
    with open(os.path.join(results_dir, 'scaler.pkl'), 'wb') as f:
        pickle.dump(scaler, f)

if kmeans_model is not None:
    with open(os.path.join(results_dir, 'kmeans_model.pkl'), 'wb') as f:
        pickle.dump(kmeans_model, f)

if msm is not None:
    with open(os.path.join(results_dir, 'msm_model.pkl'), 'wb') as f:
        pickle.dump(msm, f)

if pcca is not None:
    with open(os.path.join(results_dir, 'pcca.pkl'), 'wb') as f:
        pickle.dump(pcca, f)
# if the BayesianMSM is used, save the models
if 'models' in locals() and models is not None:
    with open(os.path.join(results_dir, 'msm_models.pkl'), 'wb') as f:
        pickle.dump(models, f)